## 1. Identificare text

In [8]:
import os
import cv2
import easyocr
import pytesseract
import Levenshtein
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

FOLDER_PATH = r"C:\Laborator AI\Laborator 2"
IMAGES = ["test1.png", "test2.jpeg"]

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")
AZURE_KEY = os.getenv("AZURE_API_KEY")

GROUND_TRUTH = {
    "test1.png": "Google Cloud Platform",
    "test2.jpeg": "Succes în rezolvarea tEMELOR la LABORAtoarele de Inteligență Artificială!"
}

def calculate_wer(reference, hypothesis): # Levenshtein
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    if not ref_words:
        return 0 if not hyp_words else 1

    return Levenshtein.distance(ref_words, hyp_words) / len(ref_words)

def calculate_cer(reference, hypothesis): # Levenshtein
    if len(reference) == 0:
        return 0 if len(hypothesis) == 0 else 1
    return Levenshtein.distance(reference, hypothesis) / len(reference)

print("Se încarcă modelele OCR...")
easyocr_reader = easyocr.Reader(['ro', 'en'], gpu=False)
azure_client = ImageAnalysisClient(endpoint=AZURE_ENDPOINT, credential=AzureKeyCredential(AZURE_KEY))

for img_name in IMAGES:
    img_path = os.path.join(FOLDER_PATH, img_name)
    if not os.path.exists(img_path):
        print(f"Avertisment: Imaginea {img_path} nu a fost găsită!")
        continue

    print(f"\n{'='*50}\nProcesare imagine: {img_name}\n{'='*50}")
    true_text = GROUND_TRUTH[img_name]

    # TESSERACT OCR
    try:
        img_cv = cv2.imread(img_path)
        tesseract_text = pytesseract.image_to_string(img_cv, lang='ron+eng').strip().replace('\n', ' ')
        tes_wer = calculate_wer(true_text, tesseract_text)
        tes_cer = calculate_cer(true_text, tesseract_text)
    except Exception as e:
        tesseract_text = f"Eroare Tesseract: {e}"
        tes_wer = 1.0
        tes_cer = 1.0

    # EASYOCR
    try:
        results_easy = easyocr_reader.readtext(img_path)
        easy_text = " ".join([text for (_, text, _) in results_easy]).strip()
        easy_wer = calculate_wer(true_text, easy_text)
        easy_cer = calculate_cer(true_text, easy_text)
    except Exception as e:
        easy_text = f"Eroare EasyOCR: {e}"
        easy_wer = 1.0
        easy_cer = 1.0

    # AZURE
    try:
        with open(img_path, "rb") as f:
            image_data = f.read()

        result_azure = azure_client.analyze(
            image_data=image_data,
            visual_features=[VisualFeatures.READ]
        )

        azure_text = ""
        if result_azure.read is not None:
            azure_text = " ".join([line.text for line in result_azure.read.blocks[0].lines]).strip()

        az_wer = calculate_wer(true_text, azure_text)
        az_cer = calculate_cer(true_text, azure_text)
    except Exception as e:
        azure_text = f"Eroare Azure: {e}"
        az_wer = 1.0
        az_cer = 1.0

    print(f"Text Real (Ground Truth):\n'{true_text}'\n")
    print(f"[Tesseract] Predicție: '{tesseract_text}'\n            WER: {tes_wer:.2%} | CER: {tes_cer:.2%}\n")
    print(f"[EasyOCR]   Predicție: '{easy_text}'\n            WER: {easy_wer:.2%} | CER: {easy_cer:.2%}\n")
    print(f"[Azure AI]  Predicție: '{azure_text}'\n            WER: {az_wer:.2%} | CER: {az_cer:.2%}\n")

Using CPU. Note: This module is much faster with a GPU.


Se încarcă modelele OCR...

Procesare imagine: test1.png


C:\Users\Ciocan Ionut\PyCharmMiscProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Text Real (Ground Truth):
'Google Cloud Platform'

[Tesseract] Predicție: ''
            WER: 100.00% | CER: 100.00%

[EasyOCR]   Predicție: 'Goog Ca P Jxform'
            WER: 133.33% | CER: 42.86%

[Azure AI]  Predicție: 'Google Cloud Platform'
            WER: 0.00% | CER: 0.00%


Procesare imagine: test2.jpeg


C:\Users\Ciocan Ionut\PyCharmMiscProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Text Real (Ground Truth):
'Succes în rezolvarea tEMELOR la LABORAtoarele de Inteligență Artificială!'

[Tesseract] Predicție: 'Laces A Ae sel UW CEMELOR TA  AP GORA Da a = ='
            WER: 144.44% | CER: 71.23%

[EasyOCR]   Predicție: 'Jucces im le 29luarea YEMELoR Ja LAcoRA teafel de Jmf @qenta cAdtificialz /'
            WER: 133.33% | CER: 42.47%

[Azure AI]  Predicție: 'Succes in rezolvarea TEMELOR la LABORA toarele de Inteligentà Artificialà!'
            WER: 66.67% | CER: 8.22%



## 2. Identificare locatie

In [1]:
import os
import cv2
import easyocr
import pytesseract
from pytesseract import Output
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

FOLDER_PATH = r"C:\Laborator AI\Laborator 2"
IMAGES = ["test1.png", "test2.jpeg"]

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")
AZURE_KEY = os.getenv("AZURE_API_KEY")

GROUND_TRUTH_BOXES = {
    "test1.png": [
        [175, 40, 420, 100],
        [235, 110, 355, 150]
    ],
    "test2.jpeg": [
        [80, 280, 1340, 480],
        [130, 580, 1050, 730],
        [70, 910, 1010, 1040],
        [100, 1120, 1450, 1300]
    ]
}

def calculate_iou(boxA, boxB): # Intersection Over Union
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    boxAArea = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    boxBArea = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)

    numitor = float(boxAArea + boxBArea - interArea)
    return interArea / numitor if numitor > 0 else 0

def evaluate_iou_for_boxes(detected_boxes, gt_boxes, model_name):
    if not detected_boxes:
        print(f"[{model_name}] Nu a detectat nicio zonă de text utilă.")
        return 0.0

    total_iou = 0
    perechi = min(len(detected_boxes), len(gt_boxes))

    for i in range(perechi):
        iou = calculate_iou(detected_boxes[i], gt_boxes[i])
        total_iou += iou

    avg_iou = total_iou / perechi if perechi > 0 else 0
    print(f"[{model_name}] Average IoU: {avg_iou:.2%}")
    return avg_iou

print("Se încarcă modelele pentru evaluarea localizării...")
easyocr_reader = easyocr.Reader(['ro', 'en'], gpu=False)
azure_client = ImageAnalysisClient(endpoint=AZURE_ENDPOINT, credential=AzureKeyCredential(AZURE_KEY))

for img_name in IMAGES:
    img_path = os.path.join(FOLDER_PATH, img_name)
    if not os.path.exists(img_path): continue

    print(f"\n{'='*55}\nComparare Localizare (IoU) pentru imaginea: {img_name}\n{'='*55}")
    gt_boxes = GROUND_TRUTH_BOXES.get(img_name, [])
    img_cv = cv2.imread(img_path)

    tesseract_boxes = []
    try:
        tes_data = pytesseract.image_to_data(img_cv, lang='ron+eng', output_type=Output.DICT)
        for i in range(len(tes_data['text'])):
            # Preluăm doar cuvintele valide (cu încredere > 0 și care nu sunt doar spații goale)
            if int(tes_data['conf'][i]) > 0 and tes_data['text'][i].strip() != '':
                x_min = tes_data['left'][i]
                y_min = tes_data['top'][i]
                x_max = x_min + tes_data['width'][i]
                y_max = y_min + tes_data['height'][i]
                tesseract_boxes.append([x_min, y_min, x_max, y_max])
    except Exception as e:
        print(f"Eroare Tesseract: {e}")

    easyocr_boxes = []
    try:
        results_easy = easyocr_reader.readtext(img_path, paragraph=True) # Improvement
        for (bbox, text) in results_easy:
            x_coords = [pt[0] for pt in bbox]
            y_coords = [pt[1] for pt in bbox]
            easyocr_boxes.append([min(x_coords), min(y_coords), max(x_coords), max(y_coords)])
    except Exception as e:
        print(f"Eroare EasyOCR: {e}")

    azure_boxes = []
    try:
        with open(img_path, "rb") as f:
            image_data = f.read()
        result_azure = azure_client.analyze(image_data=image_data, visual_features=[VisualFeatures.READ])

        if result_azure.read is not None:
            for line in result_azure.read.blocks[0].lines:
                x_coords = [point.x for point in line.bounding_polygon]
                y_coords = [point.y for point in line.bounding_polygon]
                azure_boxes.append([min(x_coords), min(y_coords), max(x_coords), max(y_coords)])
    except Exception as e:
        print(f"Eroare Azure: {e}")

    evaluate_iou_for_boxes(tesseract_boxes, gt_boxes, "Tesseract")
    evaluate_iou_for_boxes(easyocr_boxes, gt_boxes, "EasyOCR")
    evaluate_iou_for_boxes(azure_boxes, gt_boxes, "Azure AI")

Using CPU. Note: This module is much faster with a GPU.


Se încarcă modelele pentru evaluarea localizării...

Comparare Localizare (IoU) pentru imaginea: test1.png


C:\Users\Ciocan Ionut\PyCharmMiscProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Tesseract] Nu a detectat nicio zonă de text utilă.
[EasyOCR] Average IoU: 47.45%
[Azure AI] Average IoU: 90.74%

Comparare Localizare (IoU) pentru imaginea: test2.jpeg


C:\Users\Ciocan Ionut\PyCharmMiscProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Tesseract] Average IoU: 5.28%
[EasyOCR] Average IoU: 60.12%
[Azure AI] Average IoU: 89.12%


## 3. Metode de imbunatatire a recunoasterii textului

1. Imagine alb-negru pentru a lasa doar cerneala pixului
2. Imagine blurata pentru a elimina zgomotul
3. Corectarea inclinatiei